In [1]:

import json
import pennylane as qml
import pennylane.numpy as np

dev = qml.device('default.qutrit', wires = 1)

@qml.qnode(dev)
def prepare_qutrit(chi, eta):
    state = np.array([np.sin(chi)*np.cos(eta), np.sin(chi)*np.sin(eta), np.cos(chi)])
    U = np.array([[state[0], 0, 0], 
                  [state[1], 0, 0], 
                  [state[2], 0, 0]])
    U = np.eye(3)
    U[:, 0] = state
    v2 = np.array([state[1], -state[0], 0])
    v2 = v2 / np.linalg.norm(v2)
    U[:, 1] = v2
    v3 = np.cross(state, v2)
    U[:, 2] = v3 / np.linalg.norm(v3)
    qml.QutritUnitary(U, wires=0)
    return qml.probs(wires=0)

def evaluate_sum(chi, eta_array):
    sum_S = 0
    for eta in eta_array:
        prob = prepare_qutrit(chi, eta)
        sum_S += prob[2]  
    return sum_S

def run(test_case_input: str) -> str:
    ins = json.loads(test_case_input)
    outs = evaluate_sum(ins[0], ins[1])
    return str(outs)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(solution_output,expected_output, atol = 1e-4), "Not the correct sum!"

test_cases = [
    ('[0.838283, [0.6283189, 1.884956, 3.141593, 4.398230, 5.654867]]', '2.236069'),
    ('[0.4, [1.047198, 2.094395, 3.141593, 4.18879, 5.235988]]', '4.241767')
]

for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        try:
            check(output, expected_output)
            print("Correct!")
        except AssertionError as exc:
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")


Running test case 0 with input '[0.838283, [0.6283189, 1.884956, 3.141593, 4.398230, 5.654867]]'...
Correct!
Running test case 1 with input '[0.4, [1.047198, 2.094395, 3.141593, 4.18879, 5.235988]]'...
Correct!
